# 환경설정
- chroma(벡터Db) : 로컬 인메모리 모드지원, 서버불필요
- Neo4j python driver : Neo4j 서버통신
- sentence-transformers : 텍스트 임베딩
- sckit-learn : PCA 데이터 분석

In [1]:
%pip install chromadb sentence-transformers neo4j matplotlib networkx scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
# 설치 확인
import chromadb
import neo4j
import sentence_transformers
print(chromadb.__version__, neo4j.__version__, sentence_transformers.__version__)

1.5.9 6.2.0 5.5.1


### 벡터 DB
- 고차원 벡터 (Embedding) 를 저장, 유사도 검색을 수행하는데 특화
- RAG 핵심 인프라 : LLM의 외부 지식 검색에 필수적
- 메타 데이터 필터링 : 벡터 검색 + 조건 필터링 동시 지원

### 벡터 DB 종류
- Chroma : 경량             : 학습 (프로토타입)
- Pinecone : 관리형, 고가용성 : 대용량 프로젝트
- Weaviate : 멀티 모달      : 복합 검색
- Milvus : 분산처리 GPU : 대용량 프로젝트
- FAISS : Meta 오픈소스     : 연구용 (벤치마크)

In [3]:
import chromadb

client = chromadb.Client() # in-memory

# client = chromadb.PersistentClient(path='./outputs')
client.heartbeat() #

1779759032797838900

### collection 생성
- 벡터들의 논리적 그룹 (RDBMS 테이블 개념)
- 거리함수 : 

In [4]:
collection = client.get_or_create_collection(
    name = 'korean_foods',
    metadata = {'hnsw:space':'cosine'}
)

print(f'컬렉션 이름 : {collection.name}')
print(f'현재 문서 수 : {collection.count()}')

컬렉션 이름 : korean_foods
현재 문서 수 : 0


In [7]:
documents = [
    "김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다.",
    "불고기는 간장 양념에 재운 소고기를 구워 먹는 한국 전통 요리입니다.",
    "비빔밥은 밥 위에 다양한 나물과 고추장을 넣고 비벼 먹는 음식입니다.",
    "된장찌개는 된장을 풀어 두부, 감자, 호박 등을 넣고 끓인 찌개입니다.",
    "삼겹살은 돼지 뱃살을 구워 쌈 채소에 싸서 먹는 인기 있는 요리입니다.",
    "떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다.",
    "냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다.",
    "잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다.",
    "갈비탕은 소갈비를 오랫동안 끓여 만든 깊은 맛의 탕 요리입니다.",
    "순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다.",
]

metadatas = [
    {"category": "찌개", "main_ingredient": "돼지고기", "spicy": True},
    {"category": "구이", "main_ingredient": "소고기", "spicy": False},
    {"category": "밥",  "main_ingredient": "채소",   "spicy": True},
    {"category": "찌개", "main_ingredient": "된장",   "spicy": False},
    {"category": "구이", "main_ingredient": "돼지고기", "spicy": False},
    {"category": "분식", "main_ingredient": "떡",     "spicy": True},
    {"category": "면",  "main_ingredient": "메밀",   "spicy": False},
    {"category": "볶음", "main_ingredient": "당면",   "spicy": False},
    {"category": "탕",  "main_ingredient": "소고기", "spicy": False},
    {"category": "찌개", "main_ingredient": "순두부", "spicy": True},
]

ids = [f'food_{i:03d}'for i in range(len(documents))]
print(f'ids = {ids}')

collection.add(
    documents = documents,
    metadatas = metadatas,
    ids = ids
)

print(f'{collection.count()}개 문서 추가 완료')
for doc_id, doc in zip(ids, documents):
    print(f'    {doc_id} : {doc}')

ids = ['food_000', 'food_001', 'food_002', 'food_003', 'food_004', 'food_005', 'food_006', 'food_007', 'food_008', 'food_009']


C:\Users\Playdata\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:10<00:00, 1.17MiB/s]


10개 문서 추가 완료
    food_000 : 김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다.
    food_001 : 불고기는 간장 양념에 재운 소고기를 구워 먹는 한국 전통 요리입니다.
    food_002 : 비빔밥은 밥 위에 다양한 나물과 고추장을 넣고 비벼 먹는 음식입니다.
    food_003 : 된장찌개는 된장을 풀어 두부, 감자, 호박 등을 넣고 끓인 찌개입니다.
    food_004 : 삼겹살은 돼지 뱃살을 구워 쌈 채소에 싸서 먹는 인기 있는 요리입니다.
    food_005 : 떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다.
    food_006 : 냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다.
    food_007 : 잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다.
    food_008 : 갈비탕은 소갈비를 오랫동안 끓여 만든 깊은 맛의 탕 요리입니다.
    food_009 : 순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다.


# 유사도 검색

In [12]:
query = '매운 국물 요리가 먹고 싶어요'

results = collection.query(
    query_texts = [query],
    n_results = 5  # TOP_5
)

for doc, meta, dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
    similarity = 1 - dist   # 코사인 거리 -> 유사도 변환
    print(doc, meta, dist)

떡볶이는 떡과 어묵을 고추장 양념에 볶아 만든 한국의 길거리 음식입니다. {'main_ingredient': '떡', 'spicy': True, 'category': '분식'} 0.15752524137496948
순두부찌개는 부드러운 순두부에 해물이나 고기를 넣어 끓인 매운 찌개입니다. {'main_ingredient': '순두부', 'spicy': True, 'category': '찌개'} 0.19012266397476196
김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다. {'spicy': True, 'category': '찌개', 'main_ingredient': '돼지고기'} 0.21276319026947021
냉면은 메밀 면을 차가운 육수에 말아 먹는 여름철 별미입니다. {'main_ingredient': '메밀', 'category': '면', 'spicy': False} 0.2229100465774536
잡채는 당면에 다양한 채소와 고기를 볶아 만든 명절 음식입니다. {'category': '볶음', 'spicy': False, 'main_ingredient': '당면'} 0.25143176317214966


In [15]:
queries = [
    "고기를 구워서 먹는 음식",
    "시원한 여름 음식 추천해주세요",
    "명절에 먹는 전통 음식",
]

# 각 질문에 대해서 top-3 문장을 출력
for query in queries:
    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    print(f'\n질문 : {query}')
    for doc, meta, dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
        similarity = 1 - dist   # 코사인 거리 -> 유사도 변환
        print(f'유사도({similarity}) 문서 : {doc[:15]}..., 메타 : {meta}')


질문 : 고기를 구워서 먹는 음식
유사도(0.7897407412528992) 문서 : 불고기는 간장 양념에 재운 ..., 메타 : {'main_ingredient': '소고기', 'spicy': False, 'category': '구이'}
유사도(0.7736005783081055) 문서 : 떡볶이는 떡과 어묵을 고추장..., 메타 : {'main_ingredient': '떡', 'category': '분식', 'spicy': True}
유사도(0.7421424388885498) 문서 : 비빔밥은 밥 위에 다양한 나..., 메타 : {'spicy': True, 'main_ingredient': '채소', 'category': '밥'}

질문 : 시원한 여름 음식 추천해주세요
유사도(0.7804622650146484) 문서 : 비빔밥은 밥 위에 다양한 나..., 메타 : {'spicy': True, 'main_ingredient': '채소', 'category': '밥'}
유사도(0.7059831023216248) 문서 : 냉면은 메밀 면을 차가운 육..., 메타 : {'main_ingredient': '메밀', 'category': '면', 'spicy': False}
유사도(0.6690664887428284) 문서 : 김치찌개는 돼지고기와 김치를..., 메타 : {'category': '찌개', 'spicy': True, 'main_ingredient': '돼지고기'}

질문 : 명절에 먹는 전통 음식
유사도(0.773314356803894) 문서 : 비빔밥은 밥 위에 다양한 나..., 메타 : {'spicy': True, 'main_ingredient': '채소', 'category': '밥'}
유사도(0.7512445449829102) 문서 : 냉면은 메밀 면을 차가운 육..., 메타 : {'main_ingredient': '메밀', 'spicy': False, 'category': '면'}
유사도(0.7327145338058472) 문서 : 떡볶이는 떡

### 메타 데이터 필터링
- 기본 벡터 유사도 검색 + 조건필터
- 문맥을 통해 유사한 문장을 찾으면서 특정 조건을 만족하는 결과

In [17]:
# 매운 음식만 검색

results = collection.query(
    query_texts = ["따뜻한 국물 요리"],
    n_results = 3, # top_5
    where = {'spicy':True} # 매운 음식만
)

for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    similarity = 1-dist # 코사인 거리 -> 유사도 변환
    print(f'유사도({similarity}) 문서 : {doc[:15]}..., 메타:{meta}')

유사도(0.7249371409416199) 문서 : 김치찌개는 돼지고기와 김치를..., 메타:{'spicy': True, 'main_ingredient': '돼지고기', 'category': '찌개'}
유사도(0.6877250671386719) 문서 : 순두부찌개는 부드러운 순두부..., 메타:{'main_ingredient': '순두부', 'category': '찌개', 'spicy': True}
유사도(0.6661773324012756) 문서 : 떡볶이는 떡과 어묵을 고추장..., 메타:{'main_ingredient': '떡', 'category': '분식', 'spicy': True}


In [19]:
# 카테고리가 찌개인 음식
# 카테고리가 구이인 음식
# 소고기 또는 돼지고기를 사용하고 매운 음식이 아닌 것

results = collection.query(
    query_texts = ["고기 종류의 음식"],
    n_results = 3, 
    where = {
            "$and":[
                { 'main_ingredient':{"$in":["소고기","돼지고기"]} },
                { 'spicy':False }
            ]
        } 
)

for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    similarity = 1-dist # 코사인 거리 -> 유사도 변환
    print(f'유사도({similarity}) 문서 : {doc[:15]}..., 메타:{meta}')

유사도(0.5191672444343567) 문서 : 불고기는 간장 양념에 재운 ..., 메타:{'main_ingredient': '소고기', 'spicy': False, 'category': '구이'}
유사도(0.5038947463035583) 문서 : 갈비탕은 소갈비를 오랫동안 ..., 메타:{'main_ingredient': '소고기', 'category': '탕', 'spicy': False}
유사도(0.42728662490844727) 문서 : 삼겹살은 돼지 뱃살을 구워 ..., 메타:{'spicy': False, 'main_ingredient': '돼지고기', 'category': '구이'}


### 커스텀 임베딩 모델 사용
- Chroma 기본 임베딩 대신 Sentence-Transformer 다국어 모델을 사용해서 한국어 검색 성능을 향상시켜보자!
- 

In [21]:
from chromadb.utils import embedding_functions
st_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
# 커스텀 임베딩 모델용 컬렉션 생성
collection_custom = client.get_or_create_collection(
    name = 'korean_foods_custom',
    embedding_function = st_ef,
    metadata = {'hnsw:space':"cosine"}
)

# 문서 추가
collection_custom.add(
    documents = documents, 
    metadatas = metadatas,
    ids = [f'custom_{i:03d}' for i in range(len(documents))]
)

# 기본 vs 커스텀 임베딩
test_query = "뜨끈한 국물이 있는 겨울 음식"
r1 = collection.query(query_texts = [test_query], n_results = 3) # 기본
r2 = collection_custom.query(query_texts = [test_query], n_results = 3) # 커스텀

def showResult(results):
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        similarity = 1-dist # 코사인 거리 -> 유사도 변환
        print(f'유사도({similarity}) 문서 : {doc[:15]}..., 메타:{meta}')

showResult(r1)
print('\n')
showResult(r2)

유사도(0.7687559127807617) 문서 : 떡볶이는 떡과 어묵을 고추장..., 메타:{'main_ingredient': '떡', 'category': '분식', 'spicy': True}
유사도(0.742857813835144) 문서 : 김치찌개는 돼지고기와 김치를..., 메타:{'category': '찌개', 'main_ingredient': '돼지고기', 'spicy': True}
유사도(0.6985085010528564) 문서 : 냉면은 메밀 면을 차가운 육..., 메타:{'category': '면', 'spicy': False, 'main_ingredient': '메밀'}


유사도(0.6097439527511597) 문서 : 잡채는 당면에 다양한 채소와..., 메타:{'spicy': False, 'main_ingredient': '당면', 'category': '볶음'}
유사도(0.5943002700805664) 문서 : 냉면은 메밀 면을 차가운 육..., 메타:{'spicy': False, 'category': '면', 'main_ingredient': '메밀'}
유사도(0.558979868888855) 문서 : 떡볶이는 떡과 어묵을 고추장..., 메타:{'spicy': True, 'main_ingredient': '떡', 'category': '분식'}


### CRUD

In [24]:
# READ  ids로 조회
result = collection.get(ids = ["food_000","food_001"])
result

{'ids': ['food_000', 'food_001'],
 'embeddings': None,
 'documents': ['김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다.',
  '불고기는 간장 양념에 재운 소고기를 구워 먹는 한국 전통 요리입니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'category': '찌개', 'main_ingredient': '돼지고기', 'spicy': True},
  {'spicy': False, 'category': '구이', 'main_ingredient': '소고기'}]}

In [25]:
# UPDATE
collection.update(
    ids = ["food_000"],
    documents = ["김치찌개는 돼지고기와 김치를 넣고 끓인 한국의 대표적인 찌개 요리입니다."],
    metadatas = [{'main_ingredient':'돼지고기', 'category':'찌개', 'spicy':True}],
)

In [26]:
print(collection.count())
collection.delete(ids = ['food_009'])
print(collection.count())

10
9


In [ ]:
# 뉴스기사검색
# 카테고리분류 - NLP(자연어 모델로 자동 분류)
# documents + meta정보 생성
# VectorDB 구축(Chroma)
# mini RAG  Retrieval:검색  Augmentation:증강  Generation:생성 